# Étude d'hyperparamètres U-Net (ISIC 2016) — v2

Ce notebook compare plusieurs configurations d'un U-Net pour la segmentation de lésions cutanées.

**Différences avec v1 :**
- La baseline est entraînée **une seule fois** puis réutilisée dans chaque comparaison
- Chaque modèle entraîné est **sauvegardé** (poids `.keras` + résultats `.json`) dans `models/`
- **Cache intelligent :** si un résultat existe déjà, l'entraînement est sauté automatiquement → on peut relancer le notebook sans tout recalculer

In [ ]:
# ── Setup : imports, pipeline, modèle paramétrable, métriques, helpers ──

import os, random, time, gc, json as _json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit

AUTOTUNE = tf.data.AUTOTUNE
SEED = 42

# ── Dossier de sauvegarde ──
SAVE_DIR = 'models'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Pipeline ──

import glob as _glob
_kaggle = _glob.glob('/kaggle/input/*/dataset_ISIC')
if _kaggle:
    _base = _kaggle[0]
else:
    _base = 'dataset_ISIC'
IMAGES_DIR = os.path.join(_base, 'ISBI2016_ISIC_Part1_Training_Data')
MASKS_DIR  = os.path.join(_base, 'ISBI2016_ISIC_Part1_Training_GroundTruth')

def load_image_mask(img_path, mask_path):
    img  = tf.image.decode_jpeg(tf.io.read_file(img_path), channels=3)
    mask = tf.image.decode_png(tf.io.read_file(mask_path), channels=1)
    img  = tf.image.convert_image_dtype(img, tf.float32)
    mask = tf.cast(mask > 127, tf.float32)
    return img, mask

def preprocess(img, mask, img_size):
    img  = tf.image.resize(img, img_size, method='bilinear')
    mask = tf.image.resize(mask, img_size, method='nearest')
    return img, mask

def augment_flip(img, mask):
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    return img, mask

def augment_heavy(img, mask):
    img, mask = augment_flip(img, mask)
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    img  = tf.image.rot90(img, k)
    mask = tf.image.rot90(mask, k)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, mask

def make_ds(img_files, mask_files, img_size=(256,256), batch_size=8, augment_fn=None):
    ds = tf.data.Dataset.from_tensor_slices((img_files, mask_files))
    ds = ds.map(lambda i, m: load_image_mask(i, m), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda i, m: preprocess(i, m, img_size), num_parallel_calls=AUTOTUNE)
    if augment_fn:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.shuffle(len(img_files)).batch(batch_size).prefetch(AUTOTUNE)
    return ds

def _compute_lesion_bin(mask_path):
    """Calcule le ratio lesion/image et retourne un bin de taille."""
    mask = tf.image.decode_png(tf.io.read_file(mask_path), channels=1)
    ratio = float(tf.reduce_mean(tf.cast(mask > 127, tf.float32)))
    if ratio < 0.01:
        return 0, ratio  # tres petite (<1%)
    elif ratio < 0.05:
        return 1, ratio  # petite (1-5%)
    elif ratio < 0.10:
        return 2, ratio  # moyenne (5-10%)
    else:
        return 3, ratio  # grande (>10%)

def _merge_rare_bins(bins, min_count=4):
    """Fusionne les bins qui ont moins de min_count échantillons avec le bin adjacent."""
    bins = bins.copy()
    unique, counts = np.unique(bins, return_counts=True)
    for b, c in zip(unique, counts):
        if c < min_count:
            # Fusionner avec le bin adjacent le plus proche qui a assez de samples
            if b > 0:
                bins[bins == b] = b - 1
            else:
                bins[bins == b] = b + 1
            print(f'  [Warning] Bin {b} ({c} samples) fusionne avec bin adjacent')
    return bins

def split_paths(scale=0.5, val_ratio=0.2, test_ratio=0.1, seed=SEED):
    """Split stratifié par taille de lésion, avec sous-échantillonnage stratifié."""
    imgs  = sorted([os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR)])
    masks = sorted([os.path.join(MASKS_DIR, f) for f in os.listdir(MASKS_DIR)])

    # Calcul des bins sur TOUT le dataset
    all_bins = []
    all_ratios = []
    print('Calcul des ratios de lesion pour stratification...')
    for mp in masks:
        b, r = _compute_lesion_bin(mp)
        all_bins.append(b)
        all_ratios.append(r)
    all_bins = np.array(all_bins)
    all_ratios = np.array(all_ratios)

    # Fusionner les bins rares pour eviter les erreurs de StratifiedShuffleSplit
    all_bins = _merge_rare_bins(all_bins)

    # Sous-échantillonnage STRATIFIÉ (si scale < 1)
    all_indices = np.arange(len(imgs))
    if scale < 1.0:
        sss_sub = StratifiedShuffleSplit(n_splits=1, train_size=scale, random_state=seed)
        keep_idx, _ = next(sss_sub.split(all_indices, all_bins))
        imgs   = [imgs[i] for i in keep_idx]
        masks  = [masks[i] for i in keep_idx]
        bins   = all_bins[keep_idx]
        ratios = all_ratios[keep_idx]
        print(f'Sous-echantillonnage stratifie : {len(keep_idx)}/{len(all_indices)} images')
        # Re-fusionner apres sous-echantillonnage (certains bins peuvent devenir rares)
        bins = _merge_rare_bins(bins)
    else:
        bins = all_bins
        ratios = all_ratios

    # Split stratifié : d'abord train+val vs test, puis train vs val
    indices = np.arange(len(imgs))
    test_size = test_ratio
    val_size_adj = val_ratio / (1 - test_ratio)  # ratio val dans le reste

    # Split 1 : trainval vs test
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(sss1.split(indices, bins))

    # Split 2 : train vs val
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size_adj, random_state=seed)
    trainval_bins = bins[trainval_idx]
    train_sub_idx, val_sub_idx = next(sss2.split(trainval_idx, trainval_bins))
    train_idx = trainval_idx[train_sub_idx]
    val_idx   = trainval_idx[val_sub_idx]

    # Vérification de la stratification
    for name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
        b = bins[idx]
        r = ratios[idx]
        dist = {0: np.sum(b==0), 1: np.sum(b==1), 2: np.sum(b==2), 3: np.sum(b==3)}
        print(f'  {name:5s} n={len(idx):3d} | <1%:{dist[0]:2d}  1-5%:{dist[1]:2d}  '
              f'5-10%:{dist[2]:2d}  >10%:{dist[3]:2d}  | ratio moyen={np.mean(r):.3f}')

    to_list = lambda idx: ([imgs[i] for i in idx], [masks[i] for i in idx])
    return to_list(train_idx), to_list(val_idx), to_list(test_idx)

# ── Métriques & losses ──

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    return (2. * inter + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def iou_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - inter
    return (inter + smooth) / (union + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def iou_loss(y_true, y_pred):
    return 1.0 - iou_coefficient(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

# ── Boundary-weighted loss ──

def _extract_boundary(y_true, kernel_size=3):
    dilated = tf.nn.max_pool2d(y_true, ksize=kernel_size, strides=1, padding='SAME')
    eroded = 1.0 - tf.nn.max_pool2d(1.0 - y_true, ksize=kernel_size, strides=1, padding='SAME')
    boundary = dilated - eroded
    return tf.clip_by_value(boundary, 0.0, 1.0)

def make_boundary_weighted_loss(w0, kernel_size=5):
    def boundary_weighted_bce_dice(y_true, y_pred):
        boundary = _extract_boundary(y_true, kernel_size)
        weight_map = 1.0 + w0 * boundary
        bce_per_pixel = K.binary_crossentropy(y_true, y_pred)
        weighted_bce = K.mean(weight_map * bce_per_pixel)
        d_loss = dice_loss(y_true, y_pred)
        return weighted_bce + d_loss
    boundary_weighted_bce_dice.__name__ = f'boundary_w{int(w0)}_bce_dice'
    return boundary_weighted_bce_dice

boundary_w5_bce_dice  = make_boundary_weighted_loss(w0=5)
boundary_w10_bce_dice = make_boundary_weighted_loss(w0=10)

METRICS = [dice_coefficient, iou_coefficient, 'binary_accuracy']

# ── Modèle paramétrable ──

def conv_block(x, filters, activation='relu', use_batchnorm=True):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
        if use_batchnorm:
            x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)
    return x

def build_unet(input_shape=(256,256,3), base_filters=64, depth=4,
               use_skip=True, dropout_rate=0.0, activation='relu',
               upsample='bilinear', use_batchnorm=True, pooling='max'):
    """
    upsample      : 'bilinear' | 'transpose'
    use_batchnorm : True | False
    pooling       : 'max' | 'avg'
    """
    inputs = layers.Input(shape=input_shape)
    skips = []
    x = inputs
    pool_layer = layers.MaxPool2D if pooling == 'max' else layers.AveragePooling2D
    for i in range(depth):
        x = conv_block(x, base_filters * (2**i), activation, use_batchnorm)
        skips.append(x)
        x = pool_layer((2,2))(x)
    x = conv_block(x, base_filters * (2**depth), activation, use_batchnorm)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    for i in reversed(range(depth)):
        filters_i = base_filters * (2**i)
        if upsample == 'transpose':
            x = layers.Conv2DTranspose(filters_i, 2, strides=2, padding='same',
                                       kernel_initializer='he_normal')(x)
        else:
            x = layers.UpSampling2D((2,2))(x)
        if use_skip:
            x = layers.Concatenate()([x, skips[i]])
        x = conv_block(x, filters_i, activation, use_batchnorm)
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(x)
    return models.Model(inputs, outputs, name=f'unet_f{base_filters}_d{depth}')

# ── Helpers sauvegarde / chargement ──

def _safe_name(name):
    return name.replace(' ', '_').replace('/', '-').replace('(', '').replace(')', '')

def _save_result(name, res):
    path = os.path.join(SAVE_DIR, f'{_safe_name(name)}.json')
    with open(path, 'w') as f:
        _json.dump(res, f)
    print(f'  Resultat sauvegarde -> {path}')

def _load_result(name):
    path = os.path.join(SAVE_DIR, f'{_safe_name(name)}.json')
    if os.path.exists(path):
        with open(path) as f:
            return _json.load(f)
    return None

CUSTOM_OBJECTS = {
    'dice_coefficient': dice_coefficient,
    'iou_coefficient': iou_coefficient,
    'dice_loss': dice_loss,
    'bce_dice_loss': bce_dice_loss,
    'iou_loss': iou_loss,
    'boundary_w5_bce_dice': boundary_w5_bce_dice,
    'boundary_w10_bce_dice': boundary_w10_bce_dice,
}

# ── Experiment runner (avec sauvegarde + cache) ──

EPOCHS = 30
PATIENCE = 8
results = []

def run_experiment(name, model, train_ds, val_ds, lr=1e-4, loss_fn=bce_dice_loss):
    cached = _load_result(name)
    if cached is not None:
        print(f'\n{"="*60}')
        print(f'  {name}  |  CACHE - charge depuis models/')
        print(f'  -> Dice={cached["best_val_dice"]:.4f}  IoU={cached["best_val_iou"]:.4f}')
        print(f'{"="*60}')
        # Eviter les doublons dans results
        if not any(r['name'] == name for r in results):
            results.append(cached)
        del model; K.clear_session(); gc.collect()
        return cached

    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=loss_fn, metrics=METRICS)
    cb = [tf.keras.callbacks.EarlyStopping(patience=PATIENCE, monitor='val_loss',
                                           restore_best_weights=True)]
    print(f'\n{"="*60}')
    print(f'  {name}  |  params: {model.count_params():,}')
    print(f'{"="*60}')
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cb, verbose=2)
    elapsed = time.time() - t0
    history_dict = {k: [float(x) for x in v] for k, v in hist.history.items()}
    best_dice = max(history_dict['val_dice_coefficient'])
    best_iou  = max(history_dict['val_iou_coefficient'])
    res = dict(name=name, best_val_dice=best_dice, best_val_iou=best_iou,
               params=model.count_params(), time_s=round(elapsed,1), history=history_dict,
               stopped_epoch=len(history_dict['loss']))
    if not any(r['name'] == name for r in results):
        results.append(res)
    print(f'  -> Dice={best_dice:.4f}  IoU={best_iou:.4f}  ({elapsed:.0f}s, {len(history_dict["loss"])} epochs)')

    model_path = os.path.join(SAVE_DIR, f'{_safe_name(name)}.keras')
    model.save(model_path)
    print(f'  Modele sauvegarde -> {model_path}')
    _save_result(name, res)

    del hist, model
    K.clear_session()
    gc.collect()
    return res

def plot_compare(exp_results, title=''):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for r in exp_results:
        h = r['history']
        axes[0].plot(h['val_dice_coefficient'], label=r['name'])
        axes[1].plot(h['val_iou_coefficient'], label=r['name'])
    axes[0].set(title=f'{title} -- Val Dice', xlabel='Epoch', ylabel='Dice')
    axes[1].set(title=f'{title} -- Val IoU', xlabel='Epoch', ylabel='IoU')
    for ax in axes: ax.legend(); ax.grid(True)
    plt.tight_layout(); plt.show()

# ── Visualisation des prédictions ──

def plot_predictions(exp_results, examples_img, examples_mask):
    n_ex = len(examples_img)
    n_models = len(exp_results)
    n_cols = 2 + n_models

    fig, axes = plt.subplots(n_ex, n_cols, figsize=(3.5 * n_cols, 3.5 * n_ex))
    if n_ex == 1:
        axes = axes[np.newaxis, :]

    for row in range(n_ex):
        img, mask = load_image_mask(examples_img[row], examples_mask[row])
        img_show = tf.image.resize(img, (256, 256))
        mask_show = tf.image.resize(mask, (256, 256), method='nearest')
        axes[row, 0].imshow(img_show.numpy())
        axes[row, 0].set_title('Image' if row == 0 else '', fontsize=10)
        axes[row, 0].axis('off')
        axes[row, 1].imshow(mask_show.numpy().squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[row, 1].set_title('Masque reel' if row == 0 else '', fontsize=10)
        axes[row, 1].axis('off')

        for col, res in enumerate(exp_results):
            model_path = os.path.join(SAVE_DIR, f'{_safe_name(res["name"])}.keras')
            model = tf.keras.models.load_model(model_path, custom_objects=CUSTOM_OBJECTS)
            input_size = model.input_shape[1:3]
            img_r = tf.image.resize(img, input_size)
            pred = model.predict(tf.expand_dims(img_r, 0), verbose=0)[0]
            pred_bin = (pred.squeeze() > 0.5).astype(np.float32)
            pred_show = tf.image.resize(pred_bin[..., np.newaxis], (256, 256), method='nearest')
            axes[row, 2 + col].imshow(pred_show.numpy().squeeze(), cmap='gray', vmin=0, vmax=1)
            axes[row, 2 + col].set_title(res['name'] if row == 0 else '', fontsize=9)
            axes[row, 2 + col].axis('off')
            del model; K.clear_session()

    plt.tight_layout(); plt.show(); gc.collect()

print('Setup OK')
print(f'Dossier de sauvegarde : {os.path.abspath(SAVE_DIR)}/')

## 0) Préparation des données et baseline unique

On entraîne la baseline **une seule fois**. Son résultat sera réutilisé dans toutes les comparaisons.

**Configuration baseline :** `base_filters=64, depth=4, skip=True, loss=bce+dice, aug=flip, size=256, dropout=0.0, batch=8`

In [ ]:
# ── Données (scale=0.5) + Baseline (entraînée UNE SEULE FOIS) ──

(train_img, train_mask), (val_img, val_mask), (test_img, test_mask) = split_paths(scale=0.5)
print(f'Train: {len(train_img)} | Val: {len(val_img)} | Test: {len(test_img)}')

# 3 exemples fixes du test set (début, milieu, fin) pour des cas hétérogènes
n_test = len(test_img)
EX_IDX = [0, n_test // 2, n_test - 1]
ex_img  = [test_img[i] for i in EX_IDX]
ex_mask = [test_mask[i] for i in EX_IDX]
print(f'Exemples de visualisation : indices {EX_IDX}')

train_ds = make_ds(train_img, train_mask, augment_fn=augment_flip)
val_ds   = make_ds(val_img, val_mask)

# Baseline : config par défaut — entraînée une seule fois
baseline = run_experiment(
    'baseline (f64/d4/skip/bce+dice)',
    build_unet(base_filters=64, depth=4, use_skip=True),
    train_ds, val_ds
)

plot_predictions([baseline], ex_img, ex_mask)

In [ ]:
# ── Vérification du split stratifié ──

def compute_lesion_ratios(mask_paths):
    """Calcule le ratio lesion/image pour chaque masque."""
    ratios = []
    for mp in mask_paths:
        mask = tf.image.decode_png(tf.io.read_file(mp), channels=1)
        mask_bin = tf.cast(mask > 127, tf.float32)
        ratio = float(tf.reduce_mean(mask_bin))
        ratios.append(ratio)
    return np.array(ratios)

train_ratios = compute_lesion_ratios(train_mask)
val_ratios   = compute_lesion_ratios(val_mask)
test_ratios  = compute_lesion_ratios(test_mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# Memes bins que la stratification : <1%, 1-5%, 5-10%, >10%
bin_edges = [0, 0.01, 0.05, 0.10, 1.0]
bin_labels = ['<1%', '1-5%', '5-10%', '>10%']
for ax, ratios, name in zip(axes, [train_ratios, val_ratios, test_ratios],
                                   ['Train', 'Val', 'Test']):
    ax.hist(ratios, bins=20, range=(0, 0.8), edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(ratios), color='red', linestyle='--', label=f'mean={np.mean(ratios):.3f}')
    ax.set(title=f'{name} (n={len(ratios)})', xlabel='Ratio lesion/image', ylabel='Nombre')
    ax.legend()
plt.suptitle('Distribution des tailles de lesion par split (stratifie)', fontsize=13)
plt.tight_layout(); plt.show()

# Catégories alignées avec les bins de stratification
print(f'Distribution par bin de stratification :')
print(f'  {"Split":5s}   {"<1%":>5s}  {"1-5%":>5s}  {"5-10%":>5s}  {">10%":>5s}  {"mean":>6s}')
for name, ratios in [('Train', train_ratios), ('Val', val_ratios), ('Test', test_ratios)]:
    b0 = np.sum(ratios < 0.01)
    b1 = np.sum((ratios >= 0.01) & (ratios < 0.05))
    b2 = np.sum((ratios >= 0.05) & (ratios < 0.10))
    b3 = np.sum(ratios >= 0.10)
    print(f'  {name:5s}   {b0:5d}  {b1:5d}  {b2:5d}  {b3:5d}  {np.mean(ratios):6.3f}')

## 1) Expérience : capacité du réseau (`base_filters`)

**Ce qu'on teste :** `base_filters = 32, 64, 128`.

`filters=64` correspond à la baseline → on ne le ré-entraîne pas, on réutilise le résultat existant.

In [ ]:
# ── Exp 1 : base_filters (32 / 64 / 128) ──
# filters=64 = baseline → on le skip

exp_filters = [baseline]  # baseline déjà entraînée
for f in [32, 128]:
    r = run_experiment(
        f'filters={f}',
        build_unet(base_filters=f, depth=4, use_skip=True),
        train_ds, val_ds
    )
    exp_filters.append(r)

plot_compare(exp_filters, 'base_filters')
plot_predictions(exp_filters, ex_img, ex_mask)

## 2) Expérience : profondeur du U-Net (`depth`)

**Ce qu'on teste :** `depth = 3, 4, 5` blocs d'encodage.

`depth=4` correspond à la baseline → réutilisé.

In [ ]:
# ── Exp 2 : depth (3 / 4 / 5 blocs encodeur) ──
# depth=4 = baseline → on le skip

exp_depth = [baseline]
for d in [3, 5]:
    r = run_experiment(
        f'depth={d}',
        build_unet(base_filters=64, depth=d, use_skip=True),
        train_ds, val_ds
    )
    exp_depth.append(r)

plot_compare(exp_depth, 'depth')
plot_predictions(exp_depth, ex_img, ex_mask)

## 3) Expérience : rôle des skip connections

**Ce qu'on teste :** modèle **avec** puis **sans** connexions skip.

`skip=True` correspond à la baseline → réutilisé.

In [ ]:
# ── Exp 3 : skip connections (avec / sans) ──
# skip=True = baseline → on le skip (sans jeu de mots)

exp_skip = [baseline]
r = run_experiment(
    'no_skip',
    build_unet(base_filters=64, depth=4, use_skip=False),
    train_ds, val_ds
)
exp_skip.append(r)

plot_compare(exp_skip, 'skip connections')
plot_predictions(exp_skip, ex_img, ex_mask)

## 4) Expérience : fonction de coût

**Ce qu'on teste :** `BCE`, `Dice`, `BCE+Dice`, `IoU`.

`bce+dice` correspond à la baseline → réutilisé.

In [ ]:
# ── Exp 4 : loss (BCE / Dice / BCE+Dice / IoU) ──
# bce+dice = baseline → on le skip

loss_fns = {
    'bce':  tf.keras.losses.binary_crossentropy,
    'dice': dice_loss,
    'iou':  iou_loss,
}

exp_loss = [baseline]  # bce+dice déjà fait
for name, lfn in loss_fns.items():
    r = run_experiment(
        f'loss={name}',
        build_unet(base_filters=64, depth=4, use_skip=True),
        train_ds, val_ds,
        loss_fn=lfn
    )
    exp_loss.append(r)

plot_compare(exp_loss, 'loss function')
plot_predictions(exp_loss, ex_img, ex_mask)

## 5) Expérience : stratégie d'augmentation

**Ce qu'on teste :** pas d'augmentation, flips simples, augmentation forte (`heavy`).

`aug=flip` correspond à la baseline → réutilisé.

In [ ]:
# ── Exp 5 : augmentation (none / flip / heavy) ──
# aug=flip = baseline → on le skip

aug_configs = {
    'aug=none':  None,
    'aug=heavy': augment_heavy,
}

exp_aug = [baseline]  # flip déjà fait
for name, aug_fn in aug_configs.items():
    ds_train = make_ds(train_img, train_mask, augment_fn=aug_fn)
    r = run_experiment(
        name,
        build_unet(base_filters=64, depth=4, use_skip=True),
        ds_train, val_ds
    )
    exp_aug.append(r)

plot_compare(exp_aug, 'augmentation')
plot_predictions(exp_aug, ex_img, ex_mask)

## 6) Expérience : résolution d'entrée

**Ce qu'on teste :** tailles d'images `128x128`, `256x256`, `384x384`.

`size=256` correspond à la baseline → réutilisé.

In [ ]:
# ── Exp 6 : image size (128 / 256 / 384) ──
# size=256 = baseline → on le skip

exp_size = [baseline]  # 256 déjà fait
size_batch = {128: 8, 384: 2}
for sz in [128, 384]:
    bs = size_batch[sz]
    ds_tr = make_ds(train_img, train_mask, img_size=(sz, sz), batch_size=bs, augment_fn=augment_flip)
    ds_vl = make_ds(val_img, val_mask, img_size=(sz, sz), batch_size=bs)
    r = run_experiment(
        f'size={sz} (bs={bs})',
        build_unet(input_shape=(sz, sz, 3), base_filters=64, depth=4, use_skip=True),
        ds_tr, ds_vl
    )
    exp_size.append(r)

plot_compare(exp_size, 'image size')
plot_predictions(exp_size, ex_img, ex_mask)

## 7) Expérience : régularisation par dropout

**Ce qu'on teste :** `dropout_rate = 0.0, 0.2, 0.5` au bottleneck.

`dropout=0.0` correspond à la baseline → réutilisé.

In [ ]:
# ── Exp 7 : dropout au bottleneck (0.0 / 0.2 / 0.5) ──
# dropout=0.0 = baseline → on le skip

exp_drop = [baseline]  # dropout=0.0 déjà fait
for dr in [0.2, 0.5]:
    r = run_experiment(
        f'dropout={dr}',
        build_unet(base_filters=64, depth=4, use_skip=True, dropout_rate=dr),
        train_ds, val_ds
    )
    exp_drop.append(r)

plot_compare(exp_drop, 'dropout')
plot_predictions(exp_drop, ex_img, ex_mask)

## 8) Expérience : taille de batch

**Ce qu'on teste :** `batch_size = 4, 8, 16`.

`batch=8` correspond à la baseline → réutilisé.

In [ ]:
# ── Exp 8 : batch size (4 / 8 / 16) ──
# batch=8 = baseline → on le skip

exp_bs = [baseline]  # batch=8 déjà fait
for bs in [4, 16]:
    ds_tr = make_ds(train_img, train_mask, batch_size=bs, augment_fn=augment_flip)
    ds_vl = make_ds(val_img, val_mask, batch_size=bs)
    r = run_experiment(
        f'batch={bs}',
        build_unet(base_filters=64, depth=4, use_skip=True),
        ds_tr, ds_vl
    )
    exp_bs.append(r)

plot_compare(exp_bs, 'batch size')
plot_predictions(exp_bs, ex_img, ex_mask)

## 9) Expérience : learning rate

**Ce qu'on teste :** `lr = 1e-3, 1e-4, 1e-5`.

Un learning rate trop haut fait osciller la loss, un trop bas ralentit la convergence.
`lr=1e-4` correspond à la baseline.

In [ ]:
# ── Exp 9 : learning rate (1e-3 / 1e-4 / 1e-5) ──
# lr=1e-4 = baseline → on le skip

exp_lr = [baseline]
for lr in [1e-3, 1e-5]:
    r = run_experiment(
        f'lr={lr}',
        build_unet(base_filters=64, depth=4, use_skip=True),
        train_ds, val_ds,
        lr=lr
    )
    exp_lr.append(r)

plot_compare(exp_lr, 'learning rate')
plot_predictions(exp_lr, ex_img, ex_mask)

## 10) Expérience : méthode d'upsampling

**Ce qu'on teste :** `bilinear` (UpSampling2D) vs `transpose` (Conv2DTranspose).

- **Bilinéaire** : interpolation simple, pas de paramètres appris. C'est la baseline.
- **Conv2DTranspose** : déconvolution apprise, le réseau apprend comment reconstruire la résolution. Plus de paramètres mais potentiellement de meilleurs contours.

In [ ]:
# ── Exp 10 : upsampling (bilinear / transpose) ──
# bilinear = baseline → on le skip

exp_up = [baseline]
r = run_experiment(
    'upsample=transpose',
    build_unet(base_filters=64, depth=4, use_skip=True, upsample='transpose'),
    train_ds, val_ds
)
exp_up.append(r)

plot_compare(exp_up, 'upsampling method')
plot_predictions(exp_up, ex_img, ex_mask)

## 11) Expérience : pondération des contours (boundary weights)

**Ce qu'on teste :** `w0 = 0` (baseline, pas de pondération), `w0 = 5`, `w0 = 10`.

On construit une **weight map** à partir du masque :
1. **Extraction du contour** : dilatation - érosion du masque (morphologie via max pooling, kernel 5x5)
2. **Weight map** : `1 + w0 * contour` → les pixels de contour pèsent `1 + w0` fois plus dans la BCE
3. **Loss** : `mean(weight_map * BCE_per_pixel) + dice_loss`

Objectif : forcer le modèle à mieux segmenter les bords des lésions, là où les erreurs sont les plus fréquentes.

In [ ]:
# ── Exp 11 : boundary weights (w0=0 / 5 / 10) ──
# w0=0 = bce_dice_loss standard = baseline → on le skip

boundary_losses = {
    'boundary_w5':  boundary_w5_bce_dice,
    'boundary_w10': boundary_w10_bce_dice,
}

exp_boundary = [baseline]  # w0=0 déjà fait
for name, lfn in boundary_losses.items():
    r = run_experiment(
        name,
        build_unet(base_filters=64, depth=4, use_skip=True),
        train_ds, val_ds,
        loss_fn=lfn
    )
    exp_boundary.append(r)

plot_compare(exp_boundary, 'boundary weights')
plot_predictions(exp_boundary, ex_img, ex_mask)

## 12) Expérience : avec et sans Batch Normalization

**Ce qu'on teste :** `use_batchnorm = True` (baseline) vs `False`.

La BatchNorm normalise les activations entre les couches. Elle stabilise l'entraînement et régularise, mais ajoute des paramètres et change le comportement entre train et inference.

In [ ]:
# ── Exp 12 : batch normalization (avec / sans) ──
# batchnorm=True = baseline → on le skip

exp_bn = [baseline]
r = run_experiment(
    'no_batchnorm',
    build_unet(base_filters=64, depth=4, use_skip=True, use_batchnorm=False),
    train_ds, val_ds
)
exp_bn.append(r)

plot_compare(exp_bn, 'batch normalization')
plot_predictions(exp_bn, ex_img, ex_mask)

## 13) Expérience : méthode de downsampling (pooling)

**Ce qu'on teste :** `MaxPool2D` (baseline) vs `AveragePooling2D`.

- **MaxPool** : conserve l'activation maximale dans chaque fenêtre → bon pour détecter contours et textures
- **AvgPool** : moyenne des activations → lisse les features, perd moins d'information globale

In [ ]:
# ── Exp 13 : pooling (max / avg) ──
# maxpool = baseline → on le skip

exp_pool = [baseline]
r = run_experiment(
    'pooling=avg',
    build_unet(base_filters=64, depth=4, use_skip=True, pooling='avg'),
    train_ds, val_ds
)
exp_pool.append(r)

plot_compare(exp_pool, 'downsampling method')
plot_predictions(exp_pool, ex_img, ex_mask)

---

## 12) Modèle final — combinaison des meilleurs hyperparamètres

### Approche
**Passe 1** (exp 1-11) : on a testé chaque hyperparamètre isolément.
**Passe 2** (cette section) : on combine les meilleurs choix en un seul modèle.

### Sélection des hyperparamètres (v1 + v2)

Basé sur les résultats de la v1 (compromis performance / complexité) :

| Paramètre | Choix | Raison |
|---|---|---|
| `base_filters` | **64** | 128 gagne ~+0.004 mais 4x plus de params |
| `depth` | **4** | 5 gagne ~+0.006 mais beaucoup plus lourd |
| `skip` | **True** | fondamental pour l'architecture UNet |
| `loss` | **dice** | +0.03 vs bce+dice, meme cout — plus gros gain |
| `augmentation` | **flip** | suffisant, heavy n'apporte rien de plus |
| `image_size` | **256** | resolution spatiale importante en segmentation medicale |
| `dropout` | **0.0** | pas de gain significatif avec 0.2 |
| `batch_size` | **8** | standard, pas de gain ailleurs |

Les exp 9-11 (lr, upsample, boundary) sont selectionnees automatiquement ci-dessous.

In [ ]:
# ── Tableau récap de toutes les expériences ──

print(f'{"Nom":<40} {"Dice":>6} {"IoU":>6} {"Params":>12} {"Temps":>7} {"Epochs":>6}')
print('-' * 85)
for r in results:
    print(f'{r["name"]:<40} {r["best_val_dice"]:>6.4f} {r["best_val_iou"]:>6.4f} '
          f'{r["params"]:>12,} {r["time_s"]:>6.0f}s {r["stopped_epoch"]:>5}')

In [ ]:
# ── Sélection automatique des exp 9-13 ──

# Exp 9 : learning rate — on prend le meilleur (pas de difference de cout)
best_lr_res = max(exp_lr, key=lambda r: r['best_val_dice'])
best_lr_name = best_lr_res['name']
lr_map = {'baseline (f64/d4/skip/bce+dice)': 1e-4, 'lr=0.001': 1e-3, 'lr=1e-05': 1e-5}
best_lr = lr_map.get(best_lr_name, 1e-4)
print(f'Exp 9  - Meilleur lr : {best_lr} (Dice={best_lr_res["best_val_dice"]:.4f})')

# Exp 10 : upsample — on prend transpose seulement si gain > 0.01
baseline_dice = baseline['best_val_dice']
transpose_res = [r for r in exp_up if 'transpose' in r['name']]
if transpose_res:
    transpose_dice = transpose_res[0]['best_val_dice']
    gain_transpose = transpose_dice - baseline_dice
    if gain_transpose > 0.01:
        best_upsample = 'transpose'
        print(f'Exp 10 - Conv2DTranspose retenu : gain={gain_transpose:+.4f} > seuil 0.01')
    else:
        best_upsample = 'bilinear'
        print(f'Exp 10 - Bilinear conserve : gain transpose={gain_transpose:+.4f} < seuil 0.01')
else:
    best_upsample = 'bilinear'

# Exp 11 : boundary weights — on prend le meilleur (pas de difference de cout)
best_bw_res = max(exp_boundary, key=lambda r: r['best_val_dice'])
best_bw_name = best_bw_res['name']
if 'w10' in best_bw_name:
    best_loss_fn = boundary_w10_bce_dice
    best_loss_label = 'boundary_w10 + dice'
elif 'w5' in best_bw_name:
    best_loss_fn = boundary_w5_bce_dice
    best_loss_label = 'boundary_w5 + dice'
else:
    best_loss_fn = dice_loss
    best_loss_label = 'dice'
print(f'Exp 11 - Meilleure loss contours : {best_bw_name} (Dice={best_bw_res["best_val_dice"]:.4f})')

# Comparer boundary weights vs dice pure (gagnante de l'exp 4)
dice_only_res = [r for r in exp_loss if r['name'] == 'loss=dice']
if dice_only_res:
    dice_only_score = dice_only_res[0]['best_val_dice']
    if best_bw_res['best_val_dice'] > dice_only_score:
        final_loss_fn = best_loss_fn
        final_loss_label = best_loss_label
        print(f'  → Loss finale : {best_loss_label} (meilleur que dice seul)')
    else:
        final_loss_fn = dice_loss
        final_loss_label = 'dice'
        print(f'  → Loss finale : dice (meilleur que boundary weights)')
else:
    final_loss_fn = best_loss_fn
    final_loss_label = best_loss_label

# Exp 12 : batch normalization — on prend le meilleur (pas de difference de cout)
best_bn_res = max(exp_bn, key=lambda r: r['best_val_dice'])
best_bn_name = best_bn_res['name']
best_batchnorm = 'no_batchnorm' not in best_bn_name
print(f'Exp 12 - BatchNorm : {"True" if best_batchnorm else "False"} '
      f'(Dice={best_bn_res["best_val_dice"]:.4f})')

# Exp 13 : pooling — on prend le meilleur (pas de difference de cout)
best_pool_res = max(exp_pool, key=lambda r: r['best_val_dice'])
best_pool_name = best_pool_res['name']
best_pooling = 'avg' if 'avg' in best_pool_name else 'max'
print(f'Exp 13 - Pooling : {best_pooling} (Dice={best_pool_res["best_val_dice"]:.4f})')

print(f'\n{"="*60}')
print(f'CONFIG MODELE FINAL :')
print(f'  filters=64, depth=4, skip=True, upsample={best_upsample}')
print(f'  loss={final_loss_label}, lr={best_lr}, aug=flip')
print(f'  batchnorm={best_batchnorm}, pooling={best_pooling}')
print(f'  size=256, dropout=0.0, batch=8')
print(f'{"="*60}')

In [ ]:
# ── Entraînement du modèle final (sur 100% du dataset) ──

# Re-split avec scale=1.0 pour utiliser toutes les données
print('Re-chargement avec 100% du dataset pour le modele final...')
(final_train_img, final_train_mask), (final_val_img, final_val_mask), (final_test_img, final_test_mask) = split_paths(scale=1.0)
print(f'Train: {len(final_train_img)} | Val: {len(final_val_img)} | Test: {len(final_test_img)}')

final_train_ds = make_ds(final_train_img, final_train_mask, augment_fn=augment_flip)
final_val_ds   = make_ds(final_val_img, final_val_mask)

final_model = build_unet(
    base_filters=64, depth=4, use_skip=True,
    upsample=best_upsample, dropout_rate=0.0,
    use_batchnorm=best_batchnorm, pooling=best_pooling
)

final = run_experiment(
    'FINAL',
    final_model,
    final_train_ds, final_val_ds,
    lr=best_lr,
    loss_fn=final_loss_fn
)

# Comparaison baseline vs final
print(f'\n{"="*60}')
print(f'  BASELINE (50%) : Dice={baseline["best_val_dice"]:.4f}  IoU={baseline["best_val_iou"]:.4f}')
print(f'  FINAL   (100%) : Dice={final["best_val_dice"]:.4f}  IoU={final["best_val_iou"]:.4f}')
gain_dice = final['best_val_dice'] - baseline['best_val_dice']
gain_iou  = final['best_val_iou']  - baseline['best_val_iou']
print(f'  GAIN           : Dice={gain_dice:+.4f}  IoU={gain_iou:+.4f}')
print(f'{"="*60}')

# Exemples de visualisation depuis le test set 100%
n_final_test = len(final_test_img)
final_ex_idx = [0, n_final_test // 2, n_final_test - 1]
final_ex_img  = [final_test_img[i] for i in final_ex_idx]
final_ex_mask = [final_test_mask[i] for i in final_ex_idx]

plot_compare([baseline, final], 'Baseline (50%) vs Final (100%)')
plot_predictions([final], final_ex_img, final_ex_mask)

## 13) Optimisation du seuil de binarisation (threshold)

Par défaut, on binarise la sortie sigmoid avec un seuil de 0.5. Mais le seuil optimal
peut être différent, surtout avec un dataset déséquilibré. On teste tous les seuils
de 0.1 à 0.9 sur le validation set et on garde celui qui maximise le Dice.

In [ ]:
# ── Optimisation du threshold sur le modèle final ──

# Charger le modèle final
final_model_path = os.path.join(SAVE_DIR, f'{_safe_name("FINAL")}.keras')
final_model = tf.keras.models.load_model(final_model_path, custom_objects=CUSTOM_OBJECTS)

# Prédictions continues (avant binarisation) sur le validation set 100%
val_ds_no_shuffle = make_ds(final_val_img, final_val_mask)
val_preds = []
val_masks_flat = []
for imgs_batch, masks_batch in val_ds_no_shuffle:
    preds = final_model.predict(imgs_batch, verbose=0)
    val_preds.append(preds)
    val_masks_flat.append(masks_batch.numpy())

val_preds = np.concatenate(val_preds, axis=0)
val_masks_flat = np.concatenate(val_masks_flat, axis=0)

# Sweep de thresholds
thresholds = np.arange(0.1, 0.95, 0.05)
dice_scores = []
for t in thresholds:
    pred_bin = (val_preds > t).astype(np.float32)
    inter = np.sum(pred_bin * val_masks_flat)
    dice = (2 * inter + 1e-6) / (np.sum(pred_bin) + np.sum(val_masks_flat) + 1e-6)
    dice_scores.append(dice)

best_idx = np.argmax(dice_scores)
best_threshold = thresholds[best_idx]
best_dice_thresh = dice_scores[best_idx]

plt.figure(figsize=(8, 4))
plt.plot(thresholds, dice_scores, 'o-', linewidth=2)
plt.axvline(0.5, color='gray', linestyle='--', label='default (0.5)')
plt.axvline(best_threshold, color='red', linestyle='--', label=f'optimal ({best_threshold:.2f})')
plt.xlabel('Threshold'); plt.ylabel('Dice'); plt.title('Dice vs Threshold (validation set 100%)')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

print(f'Threshold par defaut (0.5) : Dice={dice_scores[np.argmin(np.abs(thresholds-0.5))]:.4f}')
print(f'Threshold optimal ({best_threshold:.2f})   : Dice={best_dice_thresh:.4f}')
print(f'Gain : {best_dice_thresh - dice_scores[np.argmin(np.abs(thresholds-0.5))]:+.4f}')

del final_model; K.clear_session(); gc.collect()

## 14) Grad-CAM : visualiser où le modèle regarde

**Grad-CAM** (Gradient-weighted Class Activation Mapping) produit une heatmap
qui montre quelles zones de l'image ont le plus influencé la prédiction du modèle.

Principe :
1. On calcule les gradients de la sortie par rapport aux feature maps du bottleneck
2. On pondère chaque feature map par la moyenne de ses gradients (importance globale)
3. On somme et on applique un ReLU → heatmap qui montre les zones les plus activées

On l'applique sur les 3 exemples fixes du test set pour vérifier que le modèle
regarde bien la lésion et pas le fond.

In [ ]:
# ── Grad-CAM sur le modèle final ──

final_model = tf.keras.models.load_model(final_model_path, custom_objects=CUSTOM_OBJECTS)

# Trouver la couche du bottleneck (dernière couche avant le décodeur)
# C'est le dernier conv_block avant l'upsampling, typiquement une Activation layer
bottleneck_layer = None
for layer in final_model.layers:
    if isinstance(layer, layers.MaxPool2D) or isinstance(layer, layers.AveragePooling2D):
        continue
    if isinstance(layer, layers.UpSampling2D) or isinstance(layer, layers.Conv2DTranspose):
        break
    if isinstance(layer, layers.Activation):
        bottleneck_layer = layer

print(f'Couche bottleneck pour Grad-CAM : {bottleneck_layer.name}')

def compute_gradcam(model, img_tensor, layer_name):
    """Calcule la heatmap Grad-CAM pour une image."""
    grad_model = tf.keras.Model(
        inputs=model.input,
        outputs=[model.get_layer(layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor[tf.newaxis, ...])
        loss = tf.reduce_mean(predictions)  # score moyen de la segmentation

    grads = tape.gradient(loss, conv_outputs)
    # Poids = moyenne spatiale des gradients par canal
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    # Combinaison pondérée des feature maps
    cam = tf.reduce_sum(weights * conv_outputs, axis=-1)[0]
    cam = tf.nn.relu(cam)  # on ne garde que les activations positives
    cam = cam / (tf.reduce_max(cam) + 1e-8)  # normalisation [0, 1]
    return cam.numpy()

# Visualisation sur les 3 exemples du test set 100%
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
col_titles = ['Image', 'Masque reel', 'Prediction', 'Grad-CAM']

for row in range(3):
    img, mask = load_image_mask(final_ex_img[row], final_ex_mask[row])
    img_r = tf.image.resize(img, (256, 256))
    mask_r = tf.image.resize(mask, (256, 256), method='nearest')

    # Prédiction
    pred = final_model.predict(img_r[tf.newaxis, ...], verbose=0)[0]
    pred_bin = (pred.squeeze() > best_threshold).astype(np.float32)

    # Grad-CAM
    cam = compute_gradcam(final_model, img_r, bottleneck_layer.name)
    cam_resized = tf.image.resize(cam[..., tf.newaxis], (256, 256)).numpy().squeeze()

    # Image originale
    axes[row, 0].imshow(img_r.numpy())
    axes[row, 0].axis('off')

    # Masque réel
    axes[row, 1].imshow(mask_r.numpy().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[row, 1].axis('off')

    # Prédiction
    axes[row, 2].imshow(pred_bin, cmap='gray', vmin=0, vmax=1)
    axes[row, 2].axis('off')

    # Grad-CAM superposé sur l'image
    axes[row, 3].imshow(img_r.numpy())
    axes[row, 3].imshow(cam_resized, cmap='jet', alpha=0.4, vmin=0, vmax=1)
    axes[row, 3].axis('off')

    if row == 0:
        for j, t in enumerate(col_titles):
            axes[row, j].set_title(t, fontsize=12)

plt.suptitle('Grad-CAM -- Ou le modele regarde', fontsize=14)
plt.tight_layout(); plt.show()

del final_model; K.clear_session(); gc.collect()